## VINS Dataset: Detecting UI Elements for Accessibility 

TODO:
- write into readme: download folder from git ahead of time, put into same working directory as this project
- 



VINS dataset is an annotated dataset containing a representative collection of UI screens across two design stages: abstract wireframes and high-fidelity fully designed interfaces. All of these UIs are annotated with bounding boxes spanning different classes of UI components. We identified a total of 11 UI components with varying functionality: background images, sliding menus, pop-up windows, input fields, icons, images, texts, switches, checked views, text buttons, and page indicators. Based on our analysis and due to relatively small training instances, we combined radio buttons and checkboxes to represent the checked view class.

VINS dataset has a total of 4,800 images of UI screens of different design stages to ensure that the VINS can perform on a wider variety of design inputs. It includes the following:

- 257 images of abstract wireframes UIs

- 4,543 images of high-fidelity screens:
    - 2000 images of Android screen collected from Rico Dataset
    - 740 images of new Android screens
    - 1200 images of iphone screens
    - 603 images of UI design collected from design sharing websites


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
torchvision.disable_beta_transforms_warning()
import torchvision.transforms.v2 as transforms
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineRenderer.figure_format = 'retina'

import os
from torch.utils.data import Dataset, DataLoader
from torchvision import  utils

import zipfile
from pathlib import Path
import os
import random
import shutil

#import data_splitter

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

plt.ion()   # interactive mode

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cpu


In [4]:
log_dir = 'logs'

# Load the tensorboard extension for Jupyter
%load_ext tensorboard
# Start tensorboard and tell it where to look for logs. It will auto-update every second.
%tensorboard --logdir {log_dir}

In [3]:
pwd

'C:\\Users\\arrow\\Documents\\CSCI 378\\final_project\\CS384_FP_VINS-UI-Reader'

In [10]:
# Setup path to data folder

'''
    #(for working on weftdrive)
data_path = Path("/home/lurena/scratch/")
image_path = data_path / "VINS_dataset"
'''

data_path = Path("C:/Users/arrow/Documents/CSCI 378/final_project")
image_path = data_path / "VINS_dataset"

# If the image folder doesn't exist, download it and prepare it... 
if image_path.is_dir():
    print(f"{image_path} directory exists.") #TODO: add check for path being empty or not to unzip
else:
    print(f"Did not find {image_path} directory, creating one...")
    image_path.mkdir(parents=True, exist_ok=True)

    # Unzip VINS file
    with zipfile.ZipFile(data_path / "VINS Dataset.zip", "r") as zip_ref:
        print("Unzipping VINS file...") 
        zip_ref.extractall(image_path)


Did not find C:\Users\arrow\Documents\CSCI 378\final_project\VINS_dataset directory, creating one...
Unzipping VINS file...


In [11]:
transform = transforms.Compose([
    transforms.ToImage(),
    transforms.ConvertImageDtype(),
])

vins = torchvision.datasets.ImageFolder(image_path, transform = transform)

# Set training/test/validate sizes
train_size = 0.6
test_size = 1-(train_size+0.1) #0.1 here defines valid_size tbh
valid_size = 1-(train_size+test_size)

if train_size + test_size + valid_size > 1:
    raise ValueError("Invalid data split sizes")

train_data, test_data, valid_data = torch.utils.data.random_split(vins, [train_size, test_size, valid_size])
#print(len(vins))
#print(len(train_data), len(test_data), len(valid_data))

classes = ['bg_image', 'sld_menu','pop_up','input_fld','icon','image','txt','switch','chck_view','txt_button','page_indicator']
#radio buttons and checkboxes represented under checked view class


In [12]:
mean = []
for x, _ in vins:
    mean.append(torch.mean(x, dim=(1, 2)))
    
mean = torch.stack(mean, dim=0).mean(dim=0)
print(mean)
std = []
for x, _ in vins:
    std.append(((x - mean[:,np.newaxis,np.newaxis]) ** 2).mean(dim=(1, 2)))
std = torch.stack(std, dim=0).mean(dim=0).sqrt()
print(mean, std)

UnidentifiedImageError: cannot identify image file <_io.BufferedReader name='C:\\Users\\arrow\\Documents\\CSCI 378\\final_project\\VINS_dataset\\__MACOSX\\All Dataset\\Android\\JPEGImages\\._Android_1.jpg'>

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 8, (7, 7), stride=2, padding=3),
            nn.ReLU(),
            nn.Conv2d(8, 16, 3, groups=2, padding='same'),
            nn.MaxPool2d(2, stride=2),
            nn.Conv2d(16, 32, 3),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(32, 11)
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
count = 0

In [ ]:
def train(lr=1e-2, epochs=10, batch_size=32,sched='step',
          acc=0.5,patience=5,step_size=7,
          use_augmentation=False,
          aug_params=None):
    #patience referring to how many epochs to wait before reducing learning rate 
    #if using the step on reduceLRonPlateau mode
    #also reducing batch_size auto to 32 in an attempt to get the training to run faster

    
    device = torch.device('cpu') # my poor laptop
    train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
    valid_loader = torch.utils.data.DataLoader(valid_data, batch_size=batch_size, shuffle=False)

    network = CNN()
    network = network.to(device)

    if use_augmentation:
        # Augmentation can be achieved using classes from torchvision.transforms.v2
        augments = transforms.Compose([
            transforms.RandomHorizontalFlip(aug_params['flip_prob']),
            transforms.RandomGrayscale(aug_params['grayscale_prob']),
            transforms.ColorJitter(
                brightness=(aug_params['bright_min'], aug_params['bright_max']),
                contrast=0,
                saturation=0,
                hue=0),
            transforms.RandomCrop(
                size=32,
                padding=aug_params['shift_size'],
                fill=cifar_mean)
        ])

    global count 
    count += 1
    # tensorboard writer mainly copied from tensorboard notes
    logger = tb.SummaryWriter(
        log_dir + '/'+ str(count) + '-cnn-lr-' + str(lr) + '-epochs-' + str(epochs)+ "-sched-" + sched)
    
    #logger = tb.SummaryWriter()
    logger.add_text('lr', str(lr))
    logger.add_text('epochs', str(epochs))
    logger.add_text('batch_size', str(batch_size))
    logger.add_text('sched', sched)
    logger.add_text('acc', str(acc))
    logger.add_text('patience', str(patience))
    logger.add_text('use_augmentation', str(aug_params))

    loss = nn.CrossEntropyLoss()
    opt = optim.SGD(network.parameters(), momentum=0.9, lr=lr)

    if sched == 'step':
        scheduler = optim.lr_scheduler.StepLR(opt, step_size=step_size, gamma=0.1)
    elif sched == 'acc': #if using ReduceLROnPlateau
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=patience, mode = 'max') #mode = max modified from hw feedback
    else:
        raise RunTimeError("Unknown schedule type: " + sched)

    global_step = 0 # used by Tensorboard to track time during the run. 

    for i in range(epochs):
        network.train()
        for batch_xs, batch_ys in train_loader:
            batch_xs = batch_xs.to(device)
            batch_ys = batch_ys.to(device)
            if use_augmentation:
                batch_xs = augments(batch_xs)
            preds = network(normalize(batch_xs))
            loss_val = loss(preds, batch_ys)
            acc = (preds.argmax(dim=1) == batch_ys).float().mean()

            logger.add_scalar('loss', loss_val, global_step=global_step)
            logger.add_scalar('training accuracy', acc, global_step=global_step)
            # I also increment the global step to indicate that an iteration has passed.
            global_step += 1

            opt.zero_grad()
            loss_val.backward()
            opt.step()

        accs = []
        network.eval()
        for batch_xs, batch_ys in valid_loader:
            batch_xs = batch_xs.to(device)
            batch_ys = batch_ys.to(device)
            preds = network(normalize(batch_xs))
            accs.append((preds.argmax(dim=1) == batch_ys).float().mean())
        # using Tensorboard to track the validation accuracy as well.
        logger.add_scalar('validation accuracy', torch.tensor(accs).mean(), global_step=global_step)

        if sched == 'step':
            scheduler.step()
        elif sched == 'acc': #if using ReduceLROnPlateau
            scheduler.step(acc)